# Title

In [1]:
%config InteractiveShell.ast_node_interactivity='last_expr_or_assign'  # always print last expr.
%config InlineBackend.figure_format = 'svg'
%load_ext autoreload
%autoreload 2
%matplotlib inline

import logging
logging.basicConfig(level=logging.INFO)

In [10]:
import numpy as np
import matplotlib.pyplot as plt
import pandas
np.set_printoptions(precision=4, floatmode='fixed', suppress=True)
rng = np.random.default_rng()

Generator(PCG64) at 0x7FEFEC77AC80

In [5]:
from tsdm.datasets import KIWI_RUNS
ds = KIWI_RUNS()
# ds.clean(force=True)
ts = ds.timeseries
ds.units

INFO:tsdm.datasets.base._base:KIWI_RUNS: START cleaning dataset!
INFO:tsdm.datasets.base._base:KIWI_RUNS/timeseries already exists, skipping.
INFO:tsdm.datasets.base._base:KIWI_RUNS/metadata already exists, skipping.
INFO:tsdm.datasets.base._base:KIWI_RUNS/units already exists, skipping.
INFO:tsdm.datasets.base._base:KIWI_RUNS: DONE cleaning dataset
INFO:tsdm.datasets.base._base:KIWI_RUNS: START loading dataset!
INFO:tsdm.datasets.base._base:KIWI_RUNS/timeseries: START loading dataset!
INFO:tsdm.datasets.base._base:KIWI_RUNS/timeseries: DONE loading dataset!
INFO:tsdm.datasets.base._base:KIWI_RUNS/metadata: START loading dataset!
INFO:tsdm.datasets.base._base:KIWI_RUNS/metadata: DONE loading dataset!
INFO:tsdm.datasets.base._base:KIWI_RUNS/units: START loading dataset!
INFO:tsdm.datasets.base._base:KIWI_RUNS/units: DONE loading dataset!
INFO:tsdm.datasets.base._base:KIWI_RUNS: DONE loading dataset!


,unit,min,max,eps,sup,mean,median,std,nan,mins,maxs
variable,,,,,,,,,,,
Flow_Air,Ln/min,0.000000,10.000000,4.990000,5.000000,5.520000,5.000000,1.690,0.000,0.010,0.115
StirringSpeed,U/min,0.000000,2600.000000,100.000000,2400.000000,697.000000,0.000000,958.000,0.000,0.633,0.001
Temperature,°C,32.700001,41.400002,33.299999,41.200001,37.299999,37.400002,0.517,0.643,0.000,0.000
Acetate,g/L,0.000000,1.180000,0.000085,0.838000,0.144000,0.114000,0.123,0.996,0.019,0.001
Base,uL,0.000000,4070.000000,5.000000,4060.000000,1170.000000,742.000000,1130.000,0.961,0.005,0.000
Cumulated_feed_volume_glucose,uL,0.000000,4100.000000,3.000000,4050.000000,399.000000,182.000000,545.000,0.000,0.310,0.001
Cumulated_feed_volume_medium,uL,0.000000,5060.000000,5.880000,5040.000000,1050.000000,750.000000,1040.000,0.000,0.045,0.000
DOT,%,0.000000,100.000000,0.010000,100.000000,71.099998,82.199997,29.500,0.563,0.064,0.030
Glucose,g/L,0.000000,6.930000,0.000766,6.870000,1.320000,0.340000,1.710,0.996,0.054,0.001


In [90]:
fluo = ts[["Cumulated_feed_volume_glucose", "Cumulated_feed_volume_medium"]].astype("float32")
(fluo == 0).mean()

variable
Cumulated_feed_volume_glucose    0.00000
Cumulated_feed_volume_medium     0.04493
dtype: float64

In [91]:
from tsdm.encoders.modular import *

In [118]:
class LogEncoder(BaseEncoder):
    """Encode data on loggarithmic scale.
    
    Uses base 2 by default for lower numerical error and fast computation.
    """
    
    threshold: np.ndarray
    replacement: np.ndarray
    
    def __init__(self) -> None:
        super().__init__()
            
    def fit(self, data, /) -> None:
        assert np.all(data >= 0)
        
        mask = data == 0
        self.threshold = data[~mask].min()
        self.replacement = np.log2(self.threshold/2)
        
    def encode(self, data, /):
        # TODO: Use copy on data.
        result = data.copy()
        mask = data <= 0
        result[:] = np.where(mask, self.replacement, np.log2(data))
        return result

    def decode(self, data, /):
        result = 2 ** data
        mask = result < self.threshold
        result[:] = np.where(mask, 0, result)
        return result

In [113]:
encoder = LogEncoder()
encoder.fit(fluo)
fluo

INFO:tsdm.util.decorators._decorators:>>> Generating functional version of <function decorator at 0x7ff008823a60> <<<
INFO:tsdm.util.decorators._decorators:>>> Generating functional version of <function decorator at 0x7ff008823a60> <<<
INFO:tsdm.util.decorators._decorators:>>> Generating functional version of <function decorator at 0x7ff008823a60> <<<


variable                               Cumulated_feed_volume_glucose  \
run_id experiment_id measurement_time                                  
355    11722         0 days 00:00:19                             1.5   
                     0 days 00:00:25                             1.5   
                     0 days 00:00:26                             1.5   
                     0 days 00:00:28                             1.5   
                     0 days 00:00:29                             1.5   
...                                                              ...   
510    16871         0 days 13:27:37                          1061.0   
                     0 days 13:28:24                          1061.0   
                     0 days 13:28:33                          1061.0   
                     0 days 13:29:35                          1061.0   
                     0 days 13:29:44                          1061.0   

variable                               Cumulated_feed_volume_medium  
run_id experiment_id measurement_time                                
355    11722         0 days 00:00:19                       0.000000  
                     0 days 00:00:25                       0.000000  
                     0 days 00:00:26                       0.000000  
                     0 days 00:00:28                       0.000000  
                     0 days 00:00:29                       0.000000  
...                                                             ...  
510    16871         0 days 13:27:37                    3787.307373  
                     0 days 13:28:24                    3787.307373  
                     0 days 13:28:33                    3787.307373  
                     0 days 13:29:35                    3787.307373  
                     0 days 13:29:44                    3787.307373  

[439061 rows x 2 columns]

In [114]:
encoded = encoder.encode(fluo)

/home/rscholz/miniconda3/envs/kiwi/lib/python3.9/site-packages/pandas/core/internals/blocks.py:402: RuntimeWarning: divide by zero encountered in log2
  result = func(self.values, **kwargs)


variable                               Cumulated_feed_volume_glucose  \
run_id experiment_id measurement_time                                  
355    11722         0 days 00:00:19                        0.584962   
                     0 days 00:00:25                        0.584962   
                     0 days 00:00:26                        0.584962   
                     0 days 00:00:28                        0.584962   
                     0 days 00:00:29                        0.584962   
...                                                              ...   
510    16871         0 days 13:27:37                       10.051208   
                     0 days 13:28:24                       10.051208   
                     0 days 13:28:33                       10.051208   
                     0 days 13:29:35                       10.051208   
                     0 days 13:29:44                       10.051208   

variable                               Cumulated_feed_volume_medium  
run_id experiment_id measurement_time                                
355    11722         0 days 00:00:19                       1.556542  
                     0 days 00:00:25                       1.556542  
                     0 days 00:00:26                       1.556542  
                     0 days 00:00:28                       1.556542  
                     0 days 00:00:29                       1.556542  
...                                                             ...  
510    16871         0 days 13:27:37                      11.886957  
                     0 days 13:28:24                      11.886957  
                     0 days 13:28:33                      11.886957  
                     0 days 13:29:35                      11.886957  
                     0 days 13:29:44                      11.886957  

[439061 rows x 2 columns]

In [115]:
decoded = encoder.decode(encoded)

variable                               Cumulated_feed_volume_glucose  \
run_id experiment_id measurement_time                                  
355    11722         0 days 00:00:19                        1.500000   
                     0 days 00:00:25                        1.500000   
                     0 days 00:00:26                        1.500000   
                     0 days 00:00:28                        1.500000   
                     0 days 00:00:29                        1.500000   
...                                                              ...   
510    16871         0 days 13:27:37                     1060.999634   
                     0 days 13:28:24                     1060.999634   
                     0 days 13:28:33                     1060.999634   
                     0 days 13:29:35                     1060.999634   
                     0 days 13:29:44                     1060.999634   

variable                               Cumulated_feed_volume_medium  
run_id experiment_id measurement_time                                
355    11722         0 days 00:00:19                        0.00000  
                     0 days 00:00:25                        0.00000  
                     0 days 00:00:26                        0.00000  
                     0 days 00:00:28                        0.00000  
                     0 days 00:00:29                        0.00000  
...                                                             ...  
510    16871         0 days 13:27:37                     3787.30835  
                     0 days 13:28:24                     3787.30835  
                     0 days 13:28:33                     3787.30835  
                     0 days 13:29:35                     3787.30835  
                     0 days 13:29:44                     3787.30835  

[439061 rows x 2 columns]

In [117]:
pandas.testing.assert_frame_equal(fluo, decoded)

In [33]:
(fluo >=0).all().all().all()

True

In [34]:
np.all(fluo >=0)

True

In [45]:
fluo.min?

Signature:
fluo.min(
    axis: 'int | None | lib.NoDefault' = <no_default>,
    skipna=True,
    level=None,
    numeric_only=None,
    **kwargs,
)
Docstring:
Return the minimum of the values over the requested axis.

If you want the *index* of the minimum, use ``idxmin``. This is the equivalent of the ``numpy.ndarray`` method ``argmin``.

Parameters
----------
axis : {index (0), columns (1)}
    Axis for the function to be applied on.
skipna : bool, default True
    Exclude NA/null values when computing the result.
level : int or level name, default None
    If the axis is a MultiIndex (hierarchical), count along a
    particular level, collapsing into a Series.
numeric_only : bool, default None
    Include only float, int, boolean columns. If None, will attempt to use
    everything, then use only numeric data. Not implemented for Series.
**kwargs
    Additional keyword arguments to be passed to the function.

Returns
-------
Series or DataFrame (if level specified)

See Also
-------